# Week 16 · Notebook 2 — Serving an LLM Pipeline (GenAI Ops Sketch)
**CSCI 250 · Fall 2026**

Wrap an LLM call behind a small **API service**, add basic **monitoring** (latency + token cost), and discuss cost/latency levers. Uses **Claude** (swap in Gemini or a local model the same way). **No OpenAI.**

## 0. Install + load keys safely

In [ ]:
!pip -q install anthropic flask
import os
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    pass  # locally, set ANTHROPIC_API_KEY in your shell
print('Anthropic key set:', bool(os.environ.get('ANTHROPIC_API_KEY')))

## 1. A monitored pipeline function
Before serving, build a plain function that calls the model **and records latency + token usage**. This is the core of GenAI Ops monitoring.

In [ ]:
import time
MODEL = 'claude-haiku-4-5-20251001'  # smallest sufficient model = cheaper, faster

def pipeline(user_msg, max_tokens=200):
    try:
        import anthropic
        client = anthropic.Anthropic()
        t0 = time.time()
        msg = client.messages.create(
            model=MODEL, max_tokens=max_tokens,
            messages=[{'role': 'user', 'content': user_msg}])
        dt = time.time() - t0
        log = {'latency_s': round(dt, 2),
               'in_tokens': msg.usage.input_tokens,
               'out_tokens': msg.usage.output_tokens}
        return msg.content[0].text, log
    except Exception as e:
        return f'[no API key / offline: {e}]', {'latency_s': None}

reply, log = pipeline('Explain LoRA in one sentence.')
print('REPLY:', reply)
print('MONITOR:', log)

## 2. Estimate cost from token usage
LLMs are priced per token (input + output). Track tokens per request so cost never surprises you. (Use current published prices; values below are illustrative.)

In [ ]:
# Illustrative per-million-token prices — check the current pricing page.
PRICE_IN, PRICE_OUT = 1.00, 5.00  # $ per 1M tokens (example)

def estimate_cost(log):
    if not log.get('in_tokens'):
        return None
    return (log['in_tokens'] * PRICE_IN + log['out_tokens'] * PRICE_OUT) / 1e6

c = estimate_cost(log)
print('estimated request cost: $%.6f' % c if c is not None else 'no usage to price')

## 3. Serve it as an API (Flask)
This cell *defines* the service. In Colab you would expose it with a tunnel; on a server you run `flask run` or deploy behind gunicorn. The key idea: your pipeline + monitoring now has an HTTP endpoint other apps can call.

In [ ]:
from flask import Flask, request, jsonify
app = Flask(__name__)

@app.post('/chat')
def chat():
    user_msg = (request.json or {}).get('message', '')
    reply, log = pipeline(user_msg)
    log['cost_usd'] = estimate_cost(log)
    return jsonify({'reply': reply, 'monitor': log})

print('Flask app defined. Endpoints:', [r.rule for r in app.url_map.iter_rules()])

## 4. Test the endpoint in-process
Use Flask's test client so we don't need a live server to verify the contract.

In [ ]:
client = app.test_client()
resp = client.post('/chat', json={'message': 'Say hello in 5 words.'})
print('HTTP', resp.status_code)
print(resp.get_json())

## 5. Cost & latency levers (the GenAI Ops checklist)
- **Smallest sufficient model**; route easy queries to a cheap model (Haiku/Flash), hard ones to a strong model (Opus/Pro).
- **Cap `max_tokens`** and shorten context (let RAG fetch only what's needed).
- **Cache** repeated prompts/answers; use prompt caching for long shared context.
- **Self-hosted?** quantize and batch.
- **Always log** prompts + responses, then re-score with an eval harness (Week 17).

## 6. Self-hosted alternative (local, no key)
The same pipeline shape works with a **local** model via Ollama — privacy and per-token control, at the cost of running the hardware yourself.

In [ ]:
def local_pipeline(user_msg):
    try:
        import ollama, time
        t0 = time.time()
        out = ollama.chat(model='llama3.2',
                          messages=[{'role': 'user', 'content': user_msg}])
        return out['message']['content'], {'latency_s': round(time.time()-t0, 2)}
    except Exception as e:
        return f'[Ollama not running: {e}]', {}

print(local_pipeline('Explain quantization in one sentence.'))